In [38]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings, ChatOllama

llm = ChatOllama(model="ornith-1.5:9b", temperature=0.4)
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe:latest")

In [39]:
from langchain_community.document_loaders import TextLoader
import json

with open("booklist.json", "r") as f:
    booklist = json.load(f)

documents = []
for book in booklist:
    loader = TextLoader(book["path"], encoding="utf-8")
    doc = loader.load()[0]
    doc.metadata["title"] = book["title"]
    doc.metadata["author"] = book["author"]
    doc.metadata["year"] = book["year"]
    doc.metadata["genre"] = book["genre"]
    doc.metadata["language"] = book["language"]
    documents.extend([doc])

print(f"Loaded {len(documents)} documents from {len(booklist)} books.")

Loaded 2 documents from 2 books.


In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

Split into 513 chunks.


In [41]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

In [42]:
from langchain.tools import tool

@tool
def search_books(query: str) -> str:
    """
    Search context from the book collection in the VectorStore and return the most relevant chunks.
    Example: "Who is Alice? How does she know the Cheshire Cat? What is the story about?"
    """
    results = vectorstore.similarity_search(query, k=3)
    return "\n\n".join([f"Title: {doc.metadata['title']}\nAuthor: {doc.metadata['author']}\nYear: {doc.metadata['year']}\nGenre: {doc.metadata['genre']}\nLanguage: {doc.metadata['language']}\nContent: {doc.page_content}" for doc in results])

In [43]:
system_prompt = """
## System Prompt: Book Context Retrieval Agent

**Role**

You are a knowledgeable book assistant. When a user asks a question, you retrieve relevant context about the requested book using your available tools, then provide a helpful, 
accurate, and well-grounded answer.

**Available Tool**

You have access to a retrieval tool that returns context about books. Use it whenever a question concerns a book, its plot, characters, themes, quotes, author, publication details, or 
similar topics. Query it with enough detail to retrieve the relevant passage. If the book title is ambiguous, disambiguate before searching.

*Adapt the tool name and parameters below to match your actual interface, e.g.:*
```
search_books(query) → context
```

**Operating Principles**

1. **Ground your answers in retrieved context.** Base every factual claim on what the tool returns. Do not fabricate details, quotes, characters, or events.
2. **Be transparent about limits.** If the retrieved context does not contain the answer, say so honestly rather than guessing.
3. **Stay on topic.** Answer what was asked. Avoid unrelated summaries or plot points unless they directly support the answer.
4. **Cite your sources.** Attribute claims to the specific context or passage when helpful (e.g., "According to the retrieved passage...").
5. **Be concise and clear.** Provide the answer directly; prioritize usefulness over length.
6. **Handle multiple books gracefully.** If a query could refer to more than one book, identify which one the user means, or address both if that's what they asked for.

**Workflow**

1. **Understand the question.** Identify the specific book (if any) and what aspect is being asked about.
2. **Retrieve context.** Call the tool with a focused query. If the first result is insufficient, refine the query and search again—try synonyms, alternate titles, character names, or 
plot keywords.
3. **Evaluate the results.** Check whether the retrieved context actually addresses the question. If it's off-topic or too vague, re-query.
4. **Synthesize an answer.** Compose a response using only the relevant retrieved information, organized for clarity.
5. **Report gaps.** If the context is incomplete for what was asked, tell the user what's missing and, where reasonable, offer the closest available information.

**Handling Retrieval Outcomes**

- **Relevant context found:** Use it to answer directly.
- **Partial match:** Answer the portion the context covers, then note that other parts are unavailable.
- **No relevant context:** Clearly state that the tool did not return usable information, and avoid inventing an answer.
- **Contradictory context:** Present the competing information and, if possible, indicate which appears more consistent with the source.
- **Ambiguous query:** Ask a brief clarifying question (e.g., which book, or which element—plot, character, author) when doing so prevents a wrong answer.

**Answering Guidelines**

- Prefer direct quotes or paraphrases from the retrieved passage for factual claims.
- For subjective questions (interpretations, recommendations), you may draw on reasoning, but still anchor concrete details in context.
- If the question asks for something the context cannot support (e.g., a plot point not present, or an out-of-scope topic), respond accordingly rather than stretching the answer.

**Prohibited Behaviors**

- Do not invent books, characters, quotes, events, or details not present in the retrieved context.
- Do not claim certainty about information the context does not support.
- Do not present guesses as facts.
- Do not ignore the tool when a book question is asked.
"""

In [44]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_books],
    system_prompt=system_prompt)

In [51]:
from langchain.messages import HumanMessage, AIMessage

query = "Who is Alice? How does she know the Cheshire Cat? What is the story about?"

def ask_chat(query: str) -> AIMessage:
    stream = agent.stream_events(
        {
            "messages": [HumanMessage(content=query)]
        },
        version="v3",
    )

    last_message = None

    for snapshot in stream.values:
        latest_message = snapshot["messages"][-1]
        if latest_message.content:
            if isinstance(latest_message, HumanMessage):
                print(f"User: {latest_message.content}")
            elif isinstance(latest_message, AIMessage):
                print(f"Agent: {latest_message.pretty_repr()}")
        elif latest_message.tool_calls:
            print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

        last_message = latest_message

    return last_message

In [ ]:
last_message = ask_chat(query)

In [47]:
from IPython.display import display, Markdown
display(Markdown(last_message.text))

Based on the retrieved context from *Alice's Adventures in Wonderland* by Lewis Carroll (1865), here's what I can tell you:

## Who is Alice?

Alice is the young protagonist of this fantasy story. She's a curious, observant girl who has fallen into the strange, dream-like world of Wonderland. Throughout the story, she tries to make sense of the bizarre events and characters around her—asking questions like which way to go, what sort of people live nearby, and trying to understand the nonsensical rules of the world she's in. She also has a baby (who, notably, turns into a pig).

## How does she know the Cheshire Cat?

This is the interesting part. In the retrieved passages, Alice encounters the Cheshire Cat sitting on a bough of a tree in the woods, and she clearly already recognizes it. The key detail is when she says to the King:

> "A cat may look at a king. I've read that in some book, but I don't remember where."

This suggests Alice had **read about the Cheshire Cat in a book before** encountering it in Wonderland—she recognizes the cat as a known character and even recalls a specific line associated with it. So rather than being surprised by the cat, she treats it as a familiar figure she'd already encountered in reading.

## What is the story about?

It's a fantasy adventure in which Alice navigates the absurd, ever-shifting world of Wonderland. The story features her encounters with a cast of eccentric characters—the Cheshire Cat, the Mad Hatter, the March Hare, the King, and the Queen. A notable scene involves a croquet game that the Queen plays (chaotically, with the King demanding the Cat be "removed"). The overall tone is playful and nonsensical, with the Cat offering famously illogical advice about directions and madness.

---

**Note on limits:** The retrieved context gives me a good picture of these three questions, but it's drawn from specific passages (the opening where Alice meets the Cat, and the croquet scene). It doesn't provide a formal plot summary of the entire book, so my description of "what the story is about" is based on these scenes rather than a complete overview. If you'd like more detail on specific plot points, I can search for those.

In [52]:
query = "Describe what happens in the first meeting between Alice and the Cheshire Cat in 'Alice's Adventures in Wonderland'."

result = ask_chat(query)

User: Describe what happens in the first meeting between Alice and the Cheshire Cat in 'Alice's Adventures in Wonderland'.
Agent: ================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'ff2ae4b9-307d-4ea0-9c0e-5028b96e8178', 'name': 'search_books', 'args': {'query': 'Alice first meeting Cheshire Cat conversation Cheshire Cat grin Wonderland'}}]
Tool Calls:
  search_books (ff2ae4b9-307d-4ea0-9c0e-5028b96e8178)
 Call ID: ff2ae4b9-307d-4ea0-9c0e-5028b96e8178
  Args:
    query: Alice first meeting Cheshire Cat conversation Cheshire Cat grin Wonderland
Agent: ================================== Ai Message ==================================

[{'type': 'text', 'text': '## The First Meeting Between Alice and the Cheshire Cat\n\nThe encounter occurs as Alice wanders through the wood after deciding to set down the baby (which has now turned into a pig) and leave it. A few yards off, she is startled by seeing the Cheshire Cat sitting on a bou

In [53]:
display(Markdown(result.text))

## The First Meeting Between Alice and the Cheshire Cat

The encounter occurs as Alice wanders through the wood after deciding to set down the baby (which has now turned into a pig) and leave it. A few yards off, she is startled by seeing the Cheshire Cat sitting on a bough of a tree.

**The initial impression.** The Cat "only grinned when it saw Alice." Alice judged it to look good-natured, but she was cautious: it had "very long claws and a great many teeth," so she felt it "ought to be treated with respect."

**The greeting.** Alice calls out to it timidly, unsure whether it would like the name: "Cheshire Puss." The Cat responds by grinning just a little wider, which Alice takes as a sign of pleasure ("Come, it's pleased so far").

**The conversation.** Their exchange quickly turns philosophical and playful:

- When Alice asks which way she should go, the Cat replies, *"That depends a good deal on where you want to get to."* Since Alice says she doesn't much care where, the Cat concludes it doesn't matter which way she goes, adding that she's sure to get somewhere if she only "walk long enough."
- The Cat points out the way to the Hatter and the March Hare, noting both are "mad." When Alice protests she doesn't want to go among mad people, the Cat declares, *"we're all mad here. I'm mad. You're mad."*
- The famous argument about madness follows: the Cat claims Alice must be mad because she came to a place where everyone is mad, and then argues that since a dog growls when angry and wags its tail when pleased, but the Cat does the opposite, the Cat must be mad. Alice counters that she'd call it "purring, not growling."
- Finally, the Cat asks if Alice plays croquet with the Queen; upon learning she hasn't been invited, it says, *"You'll see me there,"* and vanishes. It reappears to ask about the baby (now a pig), and complains that Alice's sudden appearances and disappearances make one "quite giddy."

**Overall effect.** The meeting establishes the Cheshire Cat as a whimsical, enigmatic guide—curious, a little teasing, and comfortable with the strange logic of Wonderland. Its ability to grin, vanish, and reappear sets the tone for the surreal encounters to come.